# 03. Feature Engineering & Preprocessing Pipeline
## Banking Customer Churn Analytics

### Objectives
1. Engineer domain-specific financial, demographic, and behavioral features (`BalanceSalaryRatio`, `TenureAgeRatio`, `EngagementScore`, `HighRiskDemographic`).
2. Build a reusable, leak-free Scikit-Learn `ColumnTransformer` preprocessing pipeline.
3. Perform stratified train/test split and inspect transformed predictor matrices.


In [1]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath(".."))
from src.feature_engineering import BankFeatureEngineer

df = pd.read_csv("../data/processed/bank_customer_churn_clean.csv")
print(f"Loaded Raw Dataset: {df.shape}")


Loaded Raw Dataset: (10000, 14)


## 1. Domain Feature Engineering

In [2]:
fe = BankFeatureEngineer()
df_featured = fe.create_features(df)

engineered_cols = [
    "BalanceSalaryRatio", "TenureAgeRatio", "ProductTenureRatio",
    "IsZeroBalance", "AgeGroup", "CreditScoreTier",
    "EngagementScore", "HighRiskDemographic", "MultiProductWarning", "BalancePerProduct"
]

print("Sample of newly engineered features:")
df_featured[engineered_cols].head(6)


Sample of newly engineered features:


,BalanceSalaryRatio,TenureAgeRatio,ProductTenureRatio,IsZeroBalance,AgeGroup,CreditScoreTier,EngagementScore,HighRiskDemographic,MultiProductWarning,BalancePerProduct
0,0.0000,0.0476,0.3333,1,Age30_45,Fair,3.30,0,0,0.00
1,0.7447,0.0244,0.5000,0,Age30_45,Fair,2.15,0,0,83807.86
2,1.4014,0.1905,0.3333,0,Age30_45,Poor,2.20,0,1,53220.27
3,0.0000,0.0256,1.0000,1,Age30_45,Good,0.15,0,0,0.00
4,1.5870,0.0465,0.3333,0,Age30_45,Exceptional,3.30,0,0,125510.82
5,0.7596,0.1818,0.2222,0,Age30_45,Fair,2.20,0,0,56877.89


## 2. Analyzing Engineered Features vs Churn

In [3]:
print("--- Churn Rate by Generational Age Group ---")
print(df_featured.groupby("AgeGroup")["Exited"].agg(["count", "mean"]).rename(columns={"mean": "ChurnRate"}).round(3))

print("\n--- Churn Rate by High Risk Demographic Flag ---")
print(df_featured.groupby("HighRiskDemographic")["Exited"].agg(["count", "mean"]).rename(columns={"mean": "ChurnRate"}).round(3))


--- Churn Rate by Generational Age Group ---
          count  ChurnRate
AgeGroup                  
Age30_45   6019      0.144
Age46_60   1814      0.494
Over60      526      0.279
Under30    1641      0.076

--- Churn Rate by High Risk Demographic Flag ---
                     count  ChurnRate
HighRiskDemographic                  
0                     9349      0.168
1                      651      0.710


## 3. Building Scikit-Learn Preprocessing Pipeline

In [4]:
preprocessor = fe.build_preprocessor()
print("ColumnTransformer Architecture:")
print(preprocessor)


ColumnTransformer Architecture:
ColumnTransformer(transformers=[('num', RobustScaler(),
                                 ['CreditScore', 'Age', 'Tenure', 'Balance',
                                  'NumOfProducts', 'EstimatedSalary',
                                  'BalanceSalaryRatio', 'TenureAgeRatio',
                                  'ProductTenureRatio', 'EngagementScore',
                                  'BalancePerProduct']),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Geography', 'Gender', 'AgeGroup',
                                  'CreditScoreTier']),
                                ('bin', 'passthrough',
                                 ['HasCrCard', 'IsActiveMember',
                                  'IsZeroBalance', 'HighRiskDemographic',

## 4. Stratified Train/Test Split

In [5]:
X_train, X_test, y_train, y_test, feat_cols = fe.prepare_train_test_data(df, test_size=0.20, random_state=42)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"Train Churn Rate: {y_train.mean():.2%}")
print(f"Test Churn Rate:  {y_test.mean():.2%}")


Training set: 8000 samples, 20 features
Test set:     2000 samples, 20 features
Train Churn Rate: 20.38%
Test Churn Rate:  20.35%


## 5. Save Feature-Engineered Dataset

In [6]:
feat_path = os.path.abspath("../data/processed/bank_customer_churn_features.csv")
df_featured.to_csv(feat_path, index=False)
print(f"Saved feature dataset to: {feat_path}")


Saved feature dataset to: C:\Users\asus\.gemini\antigravity-ide\scratch\banking-churn-analytics\data\processed\bank_customer_churn_features.csv
